# store

> The `learning` shelf: concepts, attempts and sessions as vault documents.

This module keeps all learning data in your vault, on one shelf named `learning`.
Each concept, attempt and session is one vault document. Thus each tool that reads the vault can search it.
The other modules read and write the shelf only through the `Learning` class in this module.

In [ ]:
#| default_exp store

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

Import the tools that this module uses, and the `Vault` class of the vault.

In [ ]:
#| export
import json, re, time, uuid, hashlib, unicodedata
from pathlib import Path
from fastcore.utils import *
from vishalakshi import Vault

## The shelf

Search reads the text of a document. The `meta` JSON keeps the state.
`Vault.set_meta` changes the state and does not embed the text again.
The table gives the source URI of each kind of document.

| kind | source |
|---|---|
| `concept` | `learning://concept/<slug>` |
| `attempt` | `learning://attempt/<id>` |
| `session` | `learning://session/<id>` |

These constants name the shelf, give the start state of a concept, and list the fields that merge.

In [ ]:
#| export
SHELF = 'learning'
#: the learner state of a concept that has no attempts yet
STATE = dict(floor=0, ceiling=6, stability=0., due=None, last=None, streak=0, misses=0, seen=0, shaky=None, misconceptions=[])
#: content fields that a new source adds to, and does not replace
LISTS = ('prereqs', 'related', 'aliases', 'provenance', 'questions')
_SKIP = ('content_hash', 'pii_scanned')

`slugify` makes the key of a concept from its name. It changes a name in an Indic script, such as Devanagari, to IAST first. It changes `+`, `#` and `&` to words, so C, C++ and C# get different slugs. A name with no letters that it can read gets a short hash of the name. Thus the same name always gives the same slug. `_src` makes the source URI of a document.

In [ ]:
#| export
SYMBOLS = {'+': ' plus ', '#': ' sharp ', '&': ' and '}

def _latin(w:str) -> str:
    "`w` in IAST when it is in an Indic script such as Devanagari or Telugu; else `w`."
    if w.isascii(): return w
    from vidyut.lipi import transliterate, detect, Scheme
    d = detect(w)
    return transliterate(w, d, Scheme.Iast) if d and d not in (Scheme.Iast, Scheme.Iso15919) else w

def slugify(s:str) -> str:
    "Make a lower-case ASCII slug from `s`: `saṃdhi` and `संधि` give `samdhi`, and `C++` gives `c-plus-plus`."
    t = re.sub(r'[+#&]', lambda m: SYMBOLS[m[0]], ' '.join(map(_latin, s.split())))
    t = unicodedata.normalize('NFKD', t).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]+', '-', t.lower()).strip('-') or 'c-' + hashlib.sha1(s.encode()).hexdigest()[:8]

def _src(kind, slug): return f'learning://{kind}/{slug}'

`_union` merges two lists and keeps the order. `concept_text` makes the text that search reads.

In [ ]:
#| export
def _union(a, b):
    "Items of `a`, then the new items of `b`, in order, compared as JSON."
    seen, out = set(), []
    for x in (*(a or ()), *(b or ())):
        k = json.dumps(x, sort_keys=True, default=str)
        if k not in seen: seen.add(k); out.append(x)
    return out

def concept_text(c) -> str:
    "The searchable text of a concept: its name, statement, aliases and subject."
    al = ', '.join(c.get('aliases') or [])
    return '\n\n'.join(o for o in (c['name'], c.get('statement', ''), al and f'Also: {al}',
                                   c.get('subject') and f'Subject: {c["subject"]}') if o)

`Learning` opens one shelf of a vault. It also accepts a path, or `None` for the default vault.

In [ ]:
#| export
class Learning:
    "Concepts, attempts and sessions on one shelf of your vault."
    def __init__(self, vault=None, shelf:str=SHELF):
        if not isinstance(vault, Vault): vault = Vault(vault)
        self.vault, self.v = vault, vault.shelf(shelf)

    def _docs(self, kind, where='', args=()):
        w = 'kind=?' + (f' and ({where})' if where else '')
        return self.v.t.docs(where=w, where_args=[kind, *args])

    def _row(self, r):
        m = {k: v for k, v in json.loads(r['meta'] or '{}').items() if k not in _SKIP}
        return AttrDict({**STATE, **m, 'doc_id': r['id'], 'name': m.get('name', r['title'])})

    def __repr__(self): return f'Learning({self.v.path!r}: {len(self._docs("concept"))} concepts)'

## Read concepts

Each method returns concepts as `AttrDict`s with the learner state filled in.

In [ ]:
#| export
@patch
def concept(self:Learning, slug:str) -> AttrDict|None:
    "One concept by slug, with its learner state."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('concept', slug)]))
    return self._row(r) if r else None

@patch
def concept_by_doc(self:Learning, doc_id:str) -> AttrDict|None:
    "The concept that a search hit (its `doc_id`) belongs to, or None when the hit is not a concept."
    r = first(self.v.t.docs(where="id=? and kind='concept'", where_args=[doc_id]))
    return self._row(r) if r else None

@patch
def concepts(self:Learning, subject:str=None) -> L:
    "All concepts, or the concepts of one subject."
    rs = self._docs('concept', *(("json_extract(meta, '$.subject')=?", [subject]) if subject else ()))
    return L(rs).map(self._row)

@patch
def subjects(self:Learning) -> list:
    "The subjects that have concepts, with counts."
    return self.v.db.q(f"select json_extract(meta, '$.subject') as subject, count(*) as n from [{self.v.t.prefix}docs] "
                       "where kind='concept' group by 1 order by 2 desc")

## Write concepts

`put` adds a concept or adds to it. It keeps the learner state, merges the list fields and replaces the other fields.

In [ ]:
#| export
@patch
def put(self:Learning,
        name:str,              # the concept's name
        statement:str='',      # a true, self-contained statement of the idea
        subject:str='',        # e.g. 'sanskrit', 'jyotisha', 'ml'
        slug:str=None,         # defaults to `slugify(name)`
        prereqs=(),            # slugs this concept needs first
        related=(),            # slugs linked to it, in any direction
        aliases=(),            # other names
        provenance=(),         # where it came from: dicts such as `{'src': 'vedicreader', 'ref': ...}`
        questions=(),          # ready questions: `{'q':..., 'a':..., 'depth':...}`
        **meta) -> AttrDict:
    "Add a concept or add to one. Keep the learner state, merge the list fields and replace the other fields."
    slug = slug or slugify(name)
    old = self.concept(slug) or {}
    new = dict(name=name, statement=statement or old.get('statement', ''), subject=subject or old.get('subject', ''), slug=slug, **meta)
    for k, v in dict(prereqs=prereqs, related=related, aliases=aliases, provenance=provenance, questions=questions).items():
        new[k] = _union(old.get(k), v)
    new['prereqs'] = [p for p in new['prereqs'] if p != slug]
    m = {**STATE, **{k: v for k, v in old.items() if k != 'doc_id'}, **new}
    r = self.v.add(concept_text(m), title=name, source=_src('concept', slug), kind='concept', meta=m)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **m)
    return self.concept(slug)

`update` and `link` change one concept. `drop` removes it.

In [ ]:
#| export
@patch
def update(self:Learning, slug:str, **kv) -> AttrDict:
    "Change the state of a concept and do not embed its text again."
    c = self.concept(slug)
    if c is None: raise KeyError(f'no concept {slug!r}')
    self.v.set_meta(c.doc_id, **kv)
    return self.concept(slug)

@patch
def link(self:Learning, slug:str, prereq:str):
    "Record that `prereq` comes before `slug`."
    c = self.concept(slug)
    if c is None: raise KeyError(f'no concept {slug!r}')
    if prereq != slug and prereq not in c.prereqs: self.update(slug, prereqs=[*c.prereqs, prereq])

@patch
def drop(self:Learning, slug:str):
    "Remove a concept. Its attempts stay as a record of what happened."
    if (c := self.concept(slug)): self.v.forget(c.doc_id)

## Move old slugs

abhyasa 0.0.5 and older made slugs in a different way. A name in Devanagari got a random slug, and C, C++ and C# got the same slug, `c`.
`reslug` finds each concept that has an old slug. It moves the concept to the slug that `slugify` gives now. It also changes the slug
in each place that points to it: the links of other concepts, attempts, sessions, journeys and demos. A slug that a source sets,
such as `sa-term-karaka`, does not change.

In [ ]:
#| export
SLUG_KEYS = ('concept', 'slug', 'focus', 'prereqs', 'related', 'concepts', 'split', 'gaps', 'of')

def _old_slug(s:str) -> str:
    "The slug that abhyasa 0.0.5 made from `s`; empty where it used a random one."
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]+', '-', s.lower()).strip('-')

def _swap(x, moves:dict, key=None):
    "`x` with each slug under a key in `SLUG_KEYS` changed as `moves` says."
    if isinstance(x, dict): return {k: _swap(v, moves, k) for k, v in x.items()}
    if isinstance(x, list): return [_swap(v, moves, key) for v in x]
    return moves.get(x, x) if key in SLUG_KEYS and isinstance(x, str) else x

`reslug` finds the moves first. With `dry_run` it stops there and writes nothing.

In [ ]:
#| export
@patch
def reslug(self:Learning, dry_run:bool=False) -> dict:
    "Move each concept with an old-style slug to the slug from `slugify`, and every slug that points to it. Returns `{old: new}`."
    cs = self.concepts()
    have, moves = set(cs.attrgot('slug')), {}
    for c in cs:
        old = _old_slug(c.name)
        if not (c.slug == old or (not old and re.fullmatch(r'[0-9a-f]{8}', c.slug))) or (new := slugify(c.name)) == c.slug: continue
        base, i = new, 2
        while new in have: new, i = f'{base}-{i}', i + 1
        moves[c.slug] = new; have.add(new)
    if dry_run or not moves: return moves
    for c in cs.filter(lambda c: c.slug in moves):
        m = {**json.loads(first(self.v.t.docs(where='id=?', where_args=[c.doc_id]))['meta'] or '{}'), 'slug': moves[c.slug]}
        self.v.add(concept_text(m), title=c.name, source=_src('concept', moves[c.slug]), kind='concept', meta=m)
        self.v.forget(c.doc_id)
    for r in self.v.t.docs():
        m = json.loads(r['meta'] or '{}')
        if (n := _swap(m, moves)) != m: self.v.set_meta(r['id'], **{k: v for k, v in n.items() if v != m.get(k)})
    return moves

## Attempts

Each answer is one attempt document. Thus search can find what the learner said. The tutor can also find old questions and not ask them again.

In [ ]:
#| export
def attempt_text(a) -> str:
    "The searchable text of an attempt: the question, the answer, the think-aloud and the feedback."
    parts = [f"Question: {a['question']}", a.get('answer') and f"Answer: {a['answer']}",
             a.get('thinking') and f"Thinking: {a['thinking']}", a.get('feedback') and f"Feedback: {a['feedback']}"]
    return '\n\n'.join(p for p in parts if p)

`record` saves one attempt.

In [ ]:
#| export
@patch
def record(self:Learning, concept:str, question:str, answer:str='', thinking:str='', correct:bool=False,
           confidence:int=None, depth:int=1, kind:str='explain', hints:int=0, dont_know:bool=False, grade:int=None,
           feedback:str='', first_error:str='', misconception:str='', session:str='', **meta) -> AttrDict:
    "File one attempt as a vault document, so the learner can search what they said."
    a = AttrDict(id=uuid.uuid4().hex[:12], concept=concept, question=question, answer=answer, thinking=thinking,
                 correct=correct, confidence=confidence, depth=depth, kind=kind, hints=hints, dont_know=dont_know,
                 grade=grade, feedback=feedback, first_error=first_error, misconception=misconception,
                 session=session, at=time.time(), **meta)
    c = self.concept(concept)
    self.v.add(attempt_text(a), title=f"{c.name if c else concept}: {question[:80]}", source=_src('attempt', a.id),
               kind='attempt', meta=dict(a))
    return a

`attempts` reads them back, with the newest first.

In [ ]:
#| export
@patch
def attempts(self:Learning, concept:str=None, session:str=None, limit:int=50, since:float=None) -> L:
    "Recent attempts, newest first, for one concept or one session; with `since`, only the attempts from that time on."
    w, args = [], []
    if since is not None: w.append("json_extract(meta, '$.at')>=?"); args.append(since)
    if concept: w.append("json_extract(meta, '$.concept')=?"); args.append(concept)
    if session: w.append("json_extract(meta, '$.session')=?"); args.append(session)
    rs = L(self._docs('attempt', ' and '.join(w), args)).map(lambda r: AttrDict(json.loads(r['meta'])))
    return L(sorted(rs, key=lambda a: -a.at)[:limit]).map(lambda a: AttrDict({k: v for k, v in a.items() if k not in _SKIP}))

`mark_attempt` adds fields to a saved attempt, for example a lesson that you asked for later. `add_gap` keeps a gap that you named on a concept.

In [ ]:
#| export
@patch
def mark_attempt(self:Learning, aid:str, **kv) -> dict:
    "Merge `kv` into the meta of attempt `aid`."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('attempt', aid)]))
    if r is None: raise KeyError(f'no attempt {aid!r}')
    return self.v.set_meta(r['id'], **kv)

@patch
def add_gap(self:Learning, slug:str, gap:str):
    "Record that you said you do not know concept `gap` while you learned `slug`."
    c = self.concept(slug)
    if c is None: raise KeyError(f'no concept {slug!r}')
    if gap != slug and gap not in (c.get('gaps') or []): self.update(slug, gaps=[*(c.get('gaps') or []), gap])

## Sessions

A session summary is one document. A new summary with the same id replaces the old one.

In [ ]:
#| export
@patch
def log_session(self:Learning, sid:str, summary:str, **meta):
    "File a session, and replace an earlier one with the same id. The meta changes also when the summary text does not."
    m = dict(id=sid, summary=summary, **meta)
    r = self.v.add(summary, title=f'Session {sid}', source=_src('session', sid), kind='session', meta=m)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **m)
    return r

@patch
def session(self:Learning, sid:str) -> AttrDict|None:
    "One filed session by id, as its meta."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('session', sid)]))
    return AttrDict({**json.loads(r['meta'] or '{}'), 'doc_id': r['id']}) if r else None

@patch
def sessions(self:Learning, limit:int=100, journey:str=None) -> L:
    "The filed sessions, the most recently changed first; with `journey`, only the sessions of that journey."
    w, args = ("json_extract(meta, '$.journey')=?", [journey]) if journey else ('', [])
    rs = L(self._docs('session', w, args)).map(lambda r: AttrDict({**json.loads(r['meta'] or '{}'), 'doc_id': r['id']}))
    return L(sorted(rs, key=lambda s: -(s.get('updated') or s.get('end') or s.get('start') or 0))[:limit])

## Search and graph

`search` uses the hybrid search of the vault. `graph` gives the nodes and links for the graph view.

In [ ]:
#| export
@patch
def search(self:Learning, q:str, kind:str=None, limit:int=10) -> L:
    "Hybrid search over the shelf: concepts, attempts and sessions."
    return L(self.v.search(q, limit=limit, kind=kind))

@patch
def graph(self:Learning, subject:str=None) -> dict:
    "Nodes and links (prereq, related, gap) for a graph view. A link to a slug that has no concept is dropped."
    cs = self.concepts(subject)
    have = set(cs.attrgot('slug'))
    nodes = [dict(id=c.slug, name=c.name, subject=c.subject, floor=c.floor, ceiling=c.ceiling, due=c.due, seen=c.seen) for c in cs]
    links = [dict(source=p, target=c.slug, kind='prereq') for c in cs for p in c.prereqs if p in have]
    links += [dict(source=c.slug, target=r, kind='related') for c in cs for r in c.related if r in have and c.slug < r]
    links += [dict(source=g, target=c.slug, kind='gap') for c in cs for g in c.get('gaps') or () if g in have]
    return dict(nodes=nodes, links=links)

## A shelf for tests

Each test uses a vault in a temporary folder with the offline hashing encoder.
The tests never open the real vault of the learner.

In [ ]:
#| export
def temp_learning(**kw) -> Learning:
    "A `Learning` on a new vault in a temporary folder, with the offline hashing encoder."
    import tempfile
    return Learning(Vault(str(Path(tempfile.mkdtemp())/'vault.db'), offline=True), **kw)

## Tests

`put` makes a concept with the start state and the slug from its name.

In [ ]:
lr = temp_learning()
c = lr.put('Reciprocal rank fusion', 'Merge ranked lists by summing 1/(k + rank) for each item.', 'ml', prereqs=['ranking'], aliases=['RRF'])
test_eq(c.slug, 'reciprocal-rank-fusion')
test_eq(c.floor, 0)
test_eq(c.prereqs, ['ranking'])

`update` changes the state.

In [ ]:
c = lr.update(c.slug, floor=3, stability=2.)
test_eq((c.floor, c.stability), (3, 2.))

A second `put` keeps the learner state and merges the lists, also when the text changes.

In [ ]:
c = lr.put('Reciprocal rank fusion', 'Merge ranked lists: score is the sum of 1/(k + rank).', related=['bm25'], aliases=['RRF', 'rank fusion'])
test_eq(c.floor, 3)
test_eq(c.subject, 'ml')
test_eq(c.aliases, ['RRF', 'rank fusion'])
test_eq(c.related, ['bm25'])
test_eq(len(lr.concepts()), 1)

`concepts` and `subjects` filter by subject. `graph` keeps only the links between known concepts.

In [ ]:
lr.put('Ranking', 'Order items by a score.', 'ml')
lr.put('Sandhi', 'Sounds change where words meet.', 'sanskrit')
test_eq(len(lr.concepts('ml')), 2)
test_eq({r['subject'] for r in lr.subjects()}, {'ml', 'sanskrit'})
g = lr.graph()
test_eq(g['links'], [dict(source='ranking', target='reciprocal-rank-fusion', kind='prereq')])

`record` saves an attempt. `attempts` and `search` find it again.

In [ ]:
a = lr.record('reciprocal-rank-fusion', 'Why does RRF ignore raw scores?', answer='Because scores from different systems are not comparable.',
              thinking='bm25 and cosine have different scales', correct=True, confidence=80, depth=3)
test_eq(lr.attempts('reciprocal-rank-fusion')[0].id, a.id)
test_eq(lr.attempts('ranking'), [])
test(lr.search('scales comparable', kind='attempt'), [], lambda a,b: len(a) > 0)

`drop` removes a concept.

In [ ]:
lr.drop('ranking')
test_is(lr.concept('ranking'), None)

`log_session` saves a session. A second call with the same id replaces it. `session` reads it back.

In [ ]:
lr.log_session('s1', '1 of 2 passed.', start=1., updated=2., asked=2)
lr.log_session('s1', '1 of 2 passed.', start=1., updated=3., asked=2, ended=3.)   # the same text: the meta still changes
test_eq((lr.session('s1').updated, lr.session('s1').ended, lr.session('s1').summary), (3., 3., '1 of 2 passed.'))
test_eq(len(lr.sessions()), 1)

`sessions` lists the most recently changed session first.

In [ ]:
lr.log_session('s2', 'Started.', start=5., updated=5.)
test_eq(lr.sessions().attrgot('id'), ['s2', 's1'])
test_is(lr.session('nope'), None)

With `journey`, `sessions` reads only the sessions of that journey, in the store.

In [ ]:
lr.log_session('s3', 'A walk.', start=6., journey='j1')
test_eq(lr.sessions(journey='j1').attrgot('id'), ['s3'])
test_eq(len(lr.sessions()), 3)

`concept_by_doc` gives the concept of a search hit. It gives None for a hit that is not a concept.

In [ ]:
cl = temp_learning()
cl.put('Reciprocal rank fusion', 'Merge ranked lists by summing 1/(k + rank).', 'ml')
cl.record('reciprocal-rank-fusion', 'Why ranks?', answer='scales differ')
test_eq(cl.concept_by_doc(cl.search('rank fusion', kind='concept')[0]['doc_id']).slug, 'reciprocal-rank-fusion')
test_is(cl.concept_by_doc(cl.search('scales differ', kind='attempt')[0]['doc_id']), None)

`slugify` reads Devanagari and Telugu, keeps C, C++ and C# apart, and gives the same hash slug for the same unreadable name.

In [ ]:
test_eq([slugify(s) for s in ('saṃdhi', 'संधि', 'గుణ', 'C', 'C++', 'C#', 'R&D')], ['samdhi', 'samdhi', 'guna', 'c', 'c-plus-plus', 'c-sharp', 'r-and-d'])
test_eq(slugify('学习'), slugify('学习'))
assert slugify('学习').startswith('c-')

This test makes a vault for `reslug` as abhyasa 0.0.5 left it: C# saved as `c`, a Devanagari name with a random slug, and links, an attempt and a session that use these slugs.

In [ ]:
ol = temp_learning()
ol.put('C#', 'A language from Microsoft.', 'code', slug='c')
ol.put('संधि', 'Sounds change where words meet.', 'sanskrit', slug='a1b2c3d4')
ol.put('karaka term', 'A set slug.', 'sanskrit', slug='sa-term-karaka')
ol.put('Ranking', 'Order items by a score.', 'ml', prereqs=['c'], related=['a1b2c3d4', 'sa-term-karaka'])
ol.record('c', 'What is C#?', answer='c')
ol.log_session('s1', 'One question.', concepts=['c'], log=[dict(concept='c', question='What is C#?')], focus='a1b2c3d4')

A dry run lists the moves and changes nothing. A slug that a source sets is not in the list.

In [ ]:
test_eq(ol.reslug(dry_run=True), {'c': 'c-sharp', 'a1b2c3d4': 'samdhi'})
assert ol.concept('c') is not None and ol.concept('c-sharp') is None

The real run moves the concepts and every slug that points to them. An answer that is the text `c` stays as it is. A second run has nothing to do.

In [ ]:
test_eq(ol.reslug(), {'c': 'c-sharp', 'a1b2c3d4': 'samdhi'})
test_eq((ol.concept('c'), ol.concept('a1b2c3d4')), (None, None))
test_eq((ol.concept('c-sharp').name, ol.concept('samdhi').statement), ('C#', 'Sounds change where words meet.'))
r = ol.concept('ranking')
test_eq((r.prereqs, r.related), (['c-sharp'], ['samdhi', 'sa-term-karaka']))
a = ol.attempts('c-sharp')[0]
test_eq((a.concept, a.answer), ('c-sharp', 'c'))
s = ol.session('s1')
test_eq((s.concepts, s.log[0]['concept'], s.focus), (['c-sharp'], 'c-sharp', 'samdhi'))
test_eq(ol.reslug(), {})

`mark_attempt` changes a saved attempt. `add_gap` keeps each gap one time, and `graph` shows it as a `gap` link.

In [ ]:
gl = temp_learning()
gl.put('Functions and the neuron', 'z = w . x + b.', 'ml'); gl.put('Weighted sum', 'Add each input times its weight.', 'ml')
a = gl.record('functions-and-the-neuron', 'Why does z double?', answer='2z')
test_eq(gl.mark_attempt(a.id, taught=True)['taught'], True)
test_eq(gl.attempts('functions-and-the-neuron')[0].taught, True)
gl.add_gap('functions-and-the-neuron', 'weighted-sum'); gl.add_gap('functions-and-the-neuron', 'weighted-sum')
test_eq(gl.concept('functions-and-the-neuron').gaps, ['weighted-sum'])
assert dict(source='weighted-sum', target='functions-and-the-neuron', kind='gap') in gl.graph()['links']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()